In [1]:
from pathlib import Path
BASE = Path("/home/lka/HYBRID_SARIMA")
print(BASE.exists(), BASE)

#Import the file to read it
import pandas as pd
disocupati_trimestral = pd.read_csv(BASE / 'data.csv', 
                       index_col='Period', 
                       parse_dates=True)
print(disocupati_trimestral)

True /home/lka/HYBRID_SARIMA


             15-34 M    15-34 F   35-49 M   35-49 F   50-74 M   50-74 F  \
Period                                                                    
2004-Q1    11.720628  17.198032  4.283283  8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493  7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125  6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448  7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241  7.577546  3.440298  4.204936   
...              ...        ...       ...       ...       ...       ...   
2025-Q2    11.341464  13.635416  5.106017  7.025122  3.776334  3.889283   
2025-Q3     9.628072  11.355385  4.656108  6.239399  3.108541  3.482718   
2025-Q4    10.903735  10.928362  3.989899  5.704370  3.250742  3.129547   
2026-Q1     9.534584  10.900056  4.316380  5.611548  2.984988  3.392216   
2026-Q2    10.244865  11.855438  5.044973  6.388627  3.268385  3.460483   

           Global_Rate  

/tmp/ipykernel_8437/3099411105.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disocupati_trimestral = pd.read_csv(BASE / 'data.csv',


In [3]:
#Dividir el dataset en training, validation y test
series = [
    '15-34 M', '15-34 F',
    '35-49 M', '35-49 F',
    '50-74 M', '50-74 F',
    'Global_Rate'
]

# Ordenar cronológicamente
df = disocupati_trimestral.sort_index()

# División 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df[series].iloc[:train_end].copy()
val = df[series].iloc[train_end:val_end].copy()
test = df[series].iloc[val_end:].copy()

# Mostrar resultados para cada serie
#for serie in series:
#    print(f"\nSerie: {serie}")
#    print(f"Train: {len(train[serie])} observaciones")
#    print(f"Validation: {len(val[serie])} observaciones")
#    print(f"Test: {len(test[serie])} observaciones")

print(train)
print(val)
print(test)


             15-34 M    15-34 F   35-49 M    35-49 F   50-74 M   50-74 F  \
Period                                                                     
2004-Q1    11.720628  17.198032  4.283283   8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493   7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125   6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448   7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241   7.577546  3.440298  4.204936   
...              ...        ...       ...        ...       ...       ...   
2018-Q2    17.908050  21.321705  7.759217  10.435680  6.683032  6.283515   
2018-Q3    16.802144  19.343542  6.191831   9.120602  5.538136  5.299045   
2018-Q4    19.322895  22.132355  7.396393  10.673055  6.327076  6.189145   
2019-Q1    19.163606  21.713279  8.111591  11.250865  6.697788  6.034775   
2019-Q2    17.128382  19.347371  7.161324   9.818228  5.738406  5.356958   

           

In [ ]:
import pmdarima as pm

arima_models = {}

for col in train.columns:
    series = train[col].dropna()  # in case there are missing values
    
    print(f"\nFitting auto_arima for series: {col} ...")
    model = pm.auto_arima(
        series,
        seasonal=True,
        m=4,                 # quarterly frequency
        stepwise=True,       # speed up search
        approximation=False, # for higher accuracy
        suppress_warnings=True,
        error_action='ignore',
        trace=False
    )
    arima_models[col] = model
    
    print(f"Best ARIMA model for {col}: ARIMA{model.order} x Seasonal{model.seasonal_order} (m=4)")
    print(f"AIC: {model.aic():.2f} | BIC: {model.bic():.2f}")
    print(model)  # print first 5 residuals as a quick check
    
# You can access the fitted models later via:
# arima_models['15-34 M'], etc.



Fitting auto_arima for series: 15-34 M ...
Best ARIMA model for 15-34 M: ARIMA(0, 1, 0) x Seasonal(1, 0, 1, 4) (m=4)
AIC: 175.47 | BIC: 181.81
 ARIMA(0,1,0)(1,0,1)[4]          

Fitting auto_arima for series: 15-34 F ...
Best ARIMA model for 15-34 F: ARIMA(1, 0, 0) x Seasonal(2, 1, 0, 4) (m=4)
AIC: 175.26 | BIC: 183.50
 ARIMA(1,0,0)(2,1,0)[4]          

Fitting auto_arima for series: 35-49 M ...
Best ARIMA model for 35-49 M: ARIMA(1, 0, 0) x Seasonal(1, 1, 2, 4) (m=4)
AIC: 59.20 | BIC: 69.50
 ARIMA(1,0,0)(1,1,2)[4]          

Fitting auto_arima for series: 35-49 F ...
Best ARIMA model for 35-49 F: ARIMA(0, 1, 0) x Seasonal(1, 0, 1, 4) (m=4)
AIC: 125.28 | BIC: 131.62
 ARIMA(0,1,0)(1,0,1)[4]          

Fitting auto_arima for series: 50-74 M ...
Best ARIMA model for 50-74 M: ARIMA(2, 1, 2) x Seasonal(0, 0, 0, 4) (m=4)
AIC: 75.19 | BIC: 85.75
 ARIMA(2,1,2)(0,0,0)[4]          

Fitting auto_arima for series: 50-74 F ...
Best ARIMA model for 50-74 F: ARIMA(2, 1, 2) x Seasonal(0, 0, 0, 4) (m

In [ ]:
import itertools
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
pq=list(itertools.product(range(5),repeat=2))
d=range(3)
pdq=[(p,d_val,q) for (p,q) in pq for d_val in d]
seasonal_pdq=[(P,D,Q,4) for P,D,Q in pdq]
sarima_models,results={},[]
for col in train.columns:
    y_train=train[col].dropna().reset_index(drop=True)
    y_val=val[col].dropna().reset_index(drop=True)
    candidatos=[]
    for order in pdq:
        for seasonal_order in seasonal_pdq:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    model=SARIMAX(y_train,order=order,seasonal_order=seasonal_order,enforce_stationarity=True,enforce_invertibility=True)
                    fitted_model=model.fit(disp=False)
                if fitted_model.mle_retvals.get("converged",False) and np.isfinite(fitted_model.bic):
                    candidatos.append((fitted_model.bic,fitted_model))
            except Exception:
                continue
    sarima_models[col]=[]
    for bic,model in sorted(candidatos,key=lambda x:x[0]):
        if len(sarima_models[col])==5:
            break
        try:
            pred=np.asarray(model.forecast(steps=len(y_val)))
            fitted=np.asarray(model.fittedvalues)
            if not np.all(np.isfinite(pred)) or not np.all(np.isfinite(fitted)):
                continue
            real=y_val.to_numpy()
            values=y_train.to_numpy()
            mae=mean_absolute_error(real,pred)
            mse=mean_squared_error(real,pred)
            mask=real!=0
            mape=np.mean(np.abs((real[mask]-pred[mask])/real[mask]))*100 if mask.any() else np.nan
            scale=np.mean(np.abs(values[4:]-values[:-4])) if len(values)>4 else np.nan
            sarima_models[col].append(model)
            lb_test=acorr_ljungbox(model.resid,return_df=True)
            lb_pvalue=lb_test["lb_pvalue"].values[-1] if not lb_test.empty else np.nan
            jb_test=jarque_bera(model.resid)
            jb_pvalue=jb_test[1] if jb_test is not None else np.nan
            hetero_test=het_arch(model.resid)
            hetero_pvalue=hetero_test[1] if hetero_test is not None else np.nan
            results.append({
                "Ljung-Box (p-value)": lb_pvalue,
                "Jarque-Bera (p-value)": jb_pvalue,
                "Heteroscedasticity (p-value)": hetero_pvalue,
                "Serie":col,"Rank":len(sarima_models[col]),
                "Order":model.model.order,"Seasonal_Order":model.model.seasonal_order,
                "AIC":model.aic,"BIC":bic,"MAE":mae,
                "MSE":mse,"RMSE":np.sqrt(mse),"MAPE":mape,
                "MASE":mae/scale if np.isfinite(scale) and scale>0 else np.nan,
                "RMSEE":np.sqrt(np.mean((values-fitted)**2)),
                "R2":r2_score(real,pred) if len(real)>1 else np.nan
            })
        except (ValueError,FloatingPointError,OverflowError):
            continue
    print(f"{col} (ok)")
    
top5_sarima=pd.DataFrame(results).round(4)
(BASE/"results").mkdir(parents=True,exist_ok=True)
top5_sarima.to_csv(BASE/"results"/"top5_sarima.csv",index=False)
display(top5_sarima)

15-34 M (ok)
15-34 F (ok)
35-49 M (ok)
35-49 F (ok)
50-74 M (ok)
50-74 F (ok)
Global_Rate (ok)


,Serie,Rank,Order,Seasonal_Order,AIC,BIC,MAE,MSE,RMSE,MAPE,MASE,RMSEE,R2
0,15-34 M,1,"(3, 1, 2)","(2, 0, 1, 4)",18.0000,36.9979,61.7400,3.858834e+03,62.1195,4.051353e+02,38.0974,38.6504,-7.399607e+02
1,15-34 M,2,"(0, 1, 0)","(0, 1, 1, 4)",160.1023,164.1884,1.8727,4.504200e+00,2.1223,1.279070e+01,1.1556,1.8953,1.351000e-01
2,15-34 M,3,"(0, 1, 0)","(1, 1, 1, 4)",160.9989,167.1280,2.7996,1.162190e+01,3.4091,2.001940e+01,1.7275,1.8894,-1.231600e+00
3,15-34 M,4,"(0, 1, 0)","(0, 1, 2, 4)",161.4346,167.5637,2.6092,9.965100e+00,3.1568,1.858970e+01,1.6100,1.8907,-9.135000e-01
4,15-34 M,5,"(1, 1, 0)","(0, 1, 1, 4)",161.9804,168.1095,1.9433,4.950700e+00,2.2250,1.337970e+01,1.1991,1.8943,4.940000e-02
5,15-34 F,1,"(0, 1, 0)","(0, 1, 1, 4)",168.4573,172.5434,1.5209,3.474900e+00,1.8641,8.026400e+00,0.9468,2.6584,2.270000e-02
6,15-34 F,2,"(0, 1, 0)","(1, 1, 1, 4)",169.6670,175.7962,1.9044,4.791400e+00,2.1889,1.056950e+01,1.1856,2.6555,-3.476000e-01
7,15-34 F,3,"(0, 1, 0)","(0, 1, 2, 4)",169.7404,175.8695,1.8356,4.398100e+00,2.0972,1.011230e+01,1.1427,2.6557,-2.370000e-01
8,15-34 F,4,"(0, 0, 3)","(0, 1, 0, 4)",167.6914,175.9332,2.1718,6.755300e+00,2.5991,1.229510e+01,1.3521,4.3092,-8.999000e-01
9,15-34 F,5,"(0, 1, 1)","(0, 1, 1, 4)",170.2446,176.3738,1.4982,3.513900e+00,1.8745,7.864200e+00,0.9327,2.6581,1.170000e-02
